# Custom Object Detection using YOLOv8

This notebook demonstrates the complete workflow for training and deploying a custom object detection model using YOLOv8.

Steps covered:
1. Downloading dataset from Roboflow
2. Training a YOLOv8 model
3. Loading the trained model
4. Running real-time object detection using webcam

In [ ]:
## Import Required Libraries
#We import the libraries required for model training, dataset handling, and real-time object detection.from ultralytics import YOLO

import cv2
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from roboflow import Roboflow
import os

## Download Dataset from Roboflow

The dataset is hosted on Roboflow.  
Using the API key, we download the dataset in YOLOv8 format.

In [ ]:
# Initialize Roboflow with API key
rf = Roboflow(api_key="ta7ig7NKoVvcqQkozzVR")

# Access workspace and project
project = rf.workspace("krishnas-workspace-h4u3l").project("object-detection-venkj")

# Select dataset version
version = project.version(1)

# Download dataset in YOLOv8 format
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to object-Detection-1 in yolov8:: 100%|██████████| 1864/1864 [00:00<00:00, 2117.28it/s]


## Verify Dataset Structure

After downloading, we verify that the dataset contains the expected files such as:
- train
- valid
- test
- data.yaml

In [ ]:
dataset_path = "object-Detection-1"

print(os.listdir(dataset_path))

['data.yaml', 'README.dataset.txt', 'README.roboflow.txt', 'test', 'train', 'valid']


## Load Pretrained YOLOv8 Model

We start with a pretrained YOLOv8 nano model (`yolov8n.pt`) and fine-tune it on our custom dataset.

In [12]:
model = YOLO("yolov8n.pt")

## Model Training

The model is trained using the dataset configuration file `data.yaml`.

Key parameters:
- epochs: number of training cycles
- imgsz: image resolution
- batch: number of images per training batch

In [14]:
model.train(
    data="object-Detection-1/data.yaml",
    epochs=50,
    imgsz=640,
    batch=8
)

Ultralytics 8.4.21  Python-3.11.14 torch-2.10.0+cpu CPU (Intel Core i5-10300H 2.50GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=object-Detection-1/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspecti

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001E7C7D2ED10>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          

## Load Best Trained Model

After training, YOLO automatically saves the best performing model weights.  
We load the `best.pt` model for inference.

In [15]:
model = YOLO("runs/detect/train2/weights/best.pt")
print("Model loaded successfully")

Model loaded successfully


## Real-Time Object Detection using Webcam

We capture frames from the webcam and run inference using the trained model.  
Detected objects are displayed with bounding boxes in real time.

In [ ]:
# Start webcam
cap = cv2.VideoCapture(0)

# Check if camera opened
if not cap.isOpened():
    print("Error: Could not open webcam")
    exit()

print("Press 'q' in the video window to stop.")

while True:
    ret, frame = cap.read()

    if not ret:
        print("Failed to grab frame")
        break

    # Resize frame for faster processing
    frame = cv2.resize(frame, (640, 480))

    # Run YOLO detection
    results = model(frame, conf=0.5)

    # Draw bounding boxes
    annotated_frame = results[0].plot()

    # Show result
    cv2.imshow("Custom YOLO Detection", annotated_frame)

    # Stop if 'q' is pressed
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release resources
cap.release()
cv2.destroyAllWindows()

Press 'q' in the video window to stop.

0: 480x640 (no detections), 146.0ms
Speed: 12.0ms preprocess, 146.0ms inference, 0.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 104.6ms
Speed: 4.0ms preprocess, 104.6ms inference, 0.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 90.3ms
Speed: 1.8ms preprocess, 90.3ms inference, 0.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 188.6ms
Speed: 2.1ms preprocess, 188.6ms inference, 0.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 109.3ms
Speed: 2.4ms preprocess, 109.3ms inference, 0.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 95.6ms
Speed: 1.8ms preprocess, 95.6ms inference, 0.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 81.9ms
Speed: 2.1ms preprocess, 81.9ms inference, 0.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detec

## Conclusion

In this notebook we:

- Downloaded a dataset from Roboflow
- Trained a custom YOLOv8 model
- Loaded the trained weights
- Performed real-time object detection using a webcam

This pipeline can be extended to applications such as surveillance, safety monitoring, and smart automation systems.